# Homework — Multi-Asset Market Structure EDA

Use this notebook to explore a synthetic cross-asset dataset in the language of **the Street**.


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


trades = pd.read_csv("./data\\trades.csv.gz", parse_dates=["execution_timestamp", "trade_date"])
counterparties = pd.read_csv("./data\\counterparties.csv")
traders = pd.read_csv("./data\\traders.csv")
instruments = pd.read_csv("./data\\instruments.csv")
events = pd.read_csv("./data\\market_events.csv", parse_dates=["event_date"])

data = {
    "trades": trades,
    "counterparties": counterparties,
    "traders": traders,
    "instruments": instruments,
    "events": events
}


In [ ]:
# Summary statistics
for name, df in data.items():
    print(f"{name}\n")
    
    # Data types
    print(f"Data types:\n{df.dtypes}\n")
    
    # Broken rows
    numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns
    for num_col in numeric_cols:
        df[num_col] = pd.to_numeric(df[num_col], errors="coerce")
    
    # Duplicates
    non_id_cols = [col for col in df.columns if "_id" not in col]
    print(f"Duplicate rows: {df.duplicated(subset=non_id_cols).sum()}\n")
    
    # String standarization
    string_cols = df.select_dtypes(include=["object", "string"]).columns
    for str_col in string_cols:
        df[str_col] = df[str_col].str.lower().str.replace("-", "_")

    # Null values
    nulls = df.isna().sum()
    print(f"Null values:\n{nulls[nulls > 0]}\n")

In [ ]:
# Merged data
df_merged = trades
for name, df in data.items():
    if name == "trades":
        continue
    elif name == "counterparties":
        df_merged = df_merged.merge(counterparties, on="counterparty_id", how="left", suffixes=("", "_counterparty"))
    elif name == "traders":
        df_merged = df_merged.merge(traders, on="trader_id", how="left", suffixes=("", "_trader"))
    elif name == "instruments":
        df_merged = df_merged.merge(instruments, on="instrument_id", how="left", suffixes=("", "_instr"))
    elif name == "events":
        df_merged = df_merged.merge(events, left_on="trade_date", right_on="event_date", how="left", suffixes=("", "_event"))
    else:
        continue
suffixes = ["_counterparty", "_trader", "_instr", "_event"]

for col in df_merged.columns:
    for suffix in suffixes:
        if col.endswith(suffix):
            base_col = col.replace(suffix, "")
            if base_col in df_merged.columns:
                if df_merged[base_col].equals(df_merged[col]):
                    df_merged.drop(columns=[col], inplace=True)

print(f"Data types:\n{df_merged.dtypes}\n")

In [13]:
# Derived fields
df_merged["execution_hour"] = df_merged["execution_timestamp"].dt.hour
df_merged["execution_month"] = df_merged["execution_timestamp"].dt.month
df_merged["is_event_window"] = np.where(df_merged["event_day_flag"] == 1, "Event Window", "Non-Event Window")
df_merged["abs_pnl_1d"] = df_merged["pnl_1d_usd"].abs()
df_merged["fee_bps_estimate"] = df_merged["execution_fee_usd"] / df_merged["notional_usd"] * 10000
df_merged["risk_to_notional"] = df_merged["risk_usd"] / df_merged["notional_usd"]
df_merged["pnl_to_risk"] = df_merged["pnl_1d_usd"] / df_merged["risk_usd"]

In [ ]:
# 3.A.1
colors = ["skyblue", "salmon", "lightgreen", "moccasin"]

pa_type_by_as_cls = pd.crosstab(df_merged["participant_type"], df_merged["asset_class"])
pa_type_by_as_cls_norm = pa_type_by_as_cls.div(pa_type_by_as_cls.sum(axis=1), axis=0)
pa_type_by_as_cls_norm_sorted = pa_type_by_as_cls_norm.sort_values(by="equity", ascending=False)
pa_type_by_as_cls_norm_sorted.plot(kind="barh", stacked=True, figsize=(10, 6), color=colors)

pa_type_by_pr_type = pd.crosstab(df_merged["participant_type"], df_merged["product_type"])

plt.xticks(rotation=45)
plt.title("Distribution of Asset Classes by Participant Type")
plt.tight_layout()

print(pa_type_by_as_cls)
print("================================\n")
print(pa_type_by_pr_type)


In [ ]:
# 3.A.2
colors = ["skyblue", "salmon", "lightgreen", "moccasin"]

hf_and_ct = pd.crosstab(df_merged["participant_type"], df_merged["asset_class"]).loc[["hedge fund", "corporate treasury"]]
hf_and_ct_norm = hf_and_ct.div(hf_and_ct.sum(axis=1), axis=0)
hf_and_ct_norm_sorted = hf_and_ct_norm.sort_values(by="equity", ascending=False)
hf_and_ct_norm_sorted.plot(kind="barh", stacked=True, figsize=(10, 6), color=colors)

plt.xticks(rotation=45)
plt.title("Distribution between Hedge Funds and Corporate Treasuries by Asset Class")
plt.tight_layout()

print(hf_and_ct)


In [ ]:
# # 3.A.3
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["skyblue", "salmon", "lightgreen", "moccasin"]

as_cls_by_count = df_merged.groupby("participant_type")["trade_id"].count()
as_cls_by_notional = df_merged.groupby("participant_type")["notional_usd"].sum()
configs = [
    (as_cls_by_count, "Trade Count by Participant Type", "Trade Count"),
    (as_cls_by_notional, "Notional USD by Participant Type", "Notional USD")
]

for ax, cfg in zip(axes, configs):
    data, title, ylabel = cfg
    data.sort_values(ascending=False).plot(kind="bar", ax=ax, title=title, color=colors)
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(ax.get_xticklabels())

plt.tight_layout()
plt.show()

print(as_cls_by_count)
print("================================\n")
print(as_cls_by_notional)
print("================================\n")


In [ ]:
fig_as_cls, axes_as_cls = plt.subplots(1, 2, figsize=(14, 7))
colors = ["skyblue", "salmon", "lightgreen", "moccasin"]

# 3.B.1
m_str_by_as_cls = pd.crosstab(df_merged["market_structure"], df_merged["asset_class"])
print(m_str_by_as_cls)
print("================================\n")

# 3.B.2
venue_by_as_cls = pd.crosstab(df_merged["venue"], df_merged["asset_class"])
print(venue_by_as_cls)
print("================================\n")

configs = [
    (m_str_by_as_cls, "Distribution of Asset Classes by Market Structure", "Market Structure"),
    (venue_by_as_cls, "Distribution of Asset Classes by Venue", "Venue")
]

for ax, cfg in zip(axes_as_cls, configs):
    data, title, ylabel = cfg
    data_norm = data.div(data.sum(axis=1), axis=0)
    data_norm.sort_values(by="equity", ascending=False).plot(kind="barh", stacked=True, ax=ax, title=title, color=colors)
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.show()

# 3.B.3
stats = [
    ("spread_bps", "skyblue"),
    ("slippage_bps", "salmon"),
    ("execution_fee_usd", "lightgreen")
    ]

fig_stats, axes_stats = plt.subplots(1, 3, figsize=(18, 5))
for ax, stat in zip(axes_stats, stats):
    stat_name, color = stat
    venue_stats = df_merged.groupby("venue")[stat_name].mean()
    venue_stats.sort_values(ascending=False).plot(kind="bar", ax=ax, title=f"Average {stat_name} by Venue", color=color)
    ax.set_ylabel(f"Average {stat_name}")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    print(f"Average {stat_name} by Venue:\n{venue_stats}\n")
    print("================================\n")

plt.tight_layout()
plt.show()


In [ ]:
# 3.C
exec_quality = ["asset_class", "participant_type", "venue", "is_event_window"]

for param in exec_quality:
    print(f"Execution quality by {param}:\n")
    print(df_merged.groupby(param)[["spread_bps", "slippage_bps", "execution_fee_usd"]].mean())
    print("================================\n")

    df_exec_quality = df_merged.groupby(param)[["spread_bps", "slippage_bps", "execution_fee_usd"]].mean()

    plt.figure(figsize=(7,5))

    plt.scatter(
        df_exec_quality["spread_bps"],
        df_exec_quality["slippage_bps"],
        s=(df_exec_quality["execution_fee_usd"] / df_exec_quality["execution_fee_usd"].max()) * 1000
    )

    for txt in df_exec_quality.index:
        plt.annotate(txt, (df_exec_quality["spread_bps"][txt], df_exec_quality["slippage_bps"][txt]))

    plt.xlabel("Spread (bps)")
    plt.ylabel("Slippage (bps)")
    plt.grid(alpha=0.3)
    plt.title("Execution quality (Bubble = Execution Fee)")
    plt.show()


In [ ]:
# 3.D
df_desks = df_merged.groupby("desk").agg(
    pnl_mean=("pnl_1d_usd", "mean"),
    pnl_std=("pnl_1d_usd", "std"),
    risk_mean=("risk_usd", "mean")
).reset_index()

df_desks["risk_adjusted_pnl"] = df_desks["pnl_mean"] / df_desks["pnl_std"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    ("pnl_mean", "Avg PnL", "skyblue"),
    ("pnl_std", "PnL Volatility (Std)", "salmon"),
    ("risk_mean", "Average Risk", "lightgreen"),
    ("risk_adjusted_pnl", "Risk-adjusted PnL", "moccasin")
]

for ax, cfg in zip(axes.flatten(), configs):
    stat_name, title, color = cfg
    df_desks.sort_values(stat_name).plot(kind="barh", x="desk", y=stat_name, ax=ax, color=color)
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

configs_top = [
    ("pnl_mean", "Highest avg PnL desk"),
    ("pnl_std", "Highest PnL volatility desk"),
    ("risk_mean", "Highest risk desk"),
    ("risk_adjusted_pnl", "Highest risk-adjusted PnL desk")
]

for cfg in configs_top:
    stat_name, title = cfg
    print(f"{title}:\n")
    print(df_desks.sort_values(by=stat_name, ascending=False).head(1))
    print("================================\n")

# 3.D.2
df_merged["notional_bucket"] = pd.qcut(df_merged["notional_usd"], q=4, labels=["Small", "Medium", "Large", "X-Large"])
print(df_merged.groupby("notional_bucket")["notional_usd"].agg(["min", "max", "count"]))
print("================================\n")
print(df_merged.groupby("notional_bucket")[["spread_bps", "slippage_bps", "execution_fee_usd"]].mean())
print("================================\n")

# 3.D.3
print(df_merged.groupby("is_block_trade")[["spread_bps", "slippage_bps", "execution_fee_usd", "notional_usd", "pnl_1d_usd"]].mean())
print("================================\n")
print(pd.crosstab(df_merged["is_block_trade"], df_merged["asset_class"]))

colors = ["skyblue", "salmon", "lightgreen", "moccasin"]
blk_trade_by_as_cls = pd.crosstab(df_merged["is_block_trade"], df_merged["asset_class"])
blk_trade_by_as_cls_norm = blk_trade_by_as_cls.div(blk_trade_by_as_cls.sum(axis=1), axis=0)
blk_trade_by_as_cls_norm.plot(kind="barh", stacked=True, figsize=(10, 6), color=colors)
plt.title("Distribution of Asset Classes by Block Trade Status")
plt.tight_layout()
plt.show()


In [ ]:
# 3.E
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

exec_hour_by_id = df_merged.groupby("execution_hour")["trade_id"].count()
exec_hour_by_notional = df_merged.groupby("execution_hour")["notional_usd"].sum()

exec_month_by_id = df_merged.groupby("execution_month")["trade_id"].count()
exec_month_by_notional = df_merged.groupby("execution_month")["notional_usd"].sum()

even_window_by_id = df_merged.groupby("is_event_window")["trade_id"].count()
even_window_by_id_pct = even_window_by_id.div(even_window_by_id.sum()) * 100
even_window_by_notional = df_merged.groupby("is_event_window")["notional_usd"].sum()
even_window_by_notional_pct = even_window_by_notional.div(even_window_by_notional.sum()) * 100

config_hour = [
    (exec_hour_by_id, "Trade Count by Execution Hour", "Trade Count", "skyblue"),
    (exec_hour_by_notional, "Notional USD by Execution Hour", "Notional USD", "salmon"),
    (exec_month_by_id, "Trade Count by Execution Month", "Trade Count", "lightgreen"),
    (exec_month_by_notional, "Notional USD by Execution Month", "Notional USD", "moccasin"),
    (even_window_by_id_pct, "Trade Count by Event Window", "Percentage (%)", "pink"),
    (even_window_by_notional_pct, "Notional USD by Event Window", "Percentage (%)", "plum")
]

for ax, cfg in zip(axes.flatten(), config_hour):
    data, title, ylabel, color = cfg
    data.plot(kind="bar", ax=ax, color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    print(data)
    print("================================\n")

plt.tight_layout()
plt.show()

# 3.E.2
colors = ["skyblue", "salmon", "lightgreen", "moccasin"]
even_window_by_asset_class = pd.crosstab(df_merged["is_event_window"], df_merged["asset_class"])
even_window_by_asc_pct = even_window_by_asset_class.div(even_window_by_asset_class.sum(axis=1), axis=0) * 100
even_window_by_asc_pct.plot(kind="bar", stacked=True, figsize=(10, 6), color=colors)
plt.title("Distribution of Asset Classes by Event Window")
plt.xlabel("Event Window")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45)
print(pd.crosstab(df_merged["is_event_window"], df_merged["asset_class"]))


In [ ]:
# 3.F.1
negative_price = df_merged[df_merged["price"] < 0]
zero_quantity = df_merged[df_merged["quantity"] == 0]
negative_fee = df_merged[df_merged["execution_fee_usd"] < 0]
extreme_notional = df_merged[df_merged["notional_usd"] > df_merged["notional_usd"].quantile(0.99)]
non_id_cols = [col for col in df_merged.columns if "_id" not in col]
duplicated = df_merged[df_merged.duplicated(subset=non_id_cols)]

cols = ["trade_id", "price", "asset_class", "product_type", "quantity", "execution_fee_usd", "notional_usd"]
configs = [
    (negative_price, "Negative prices"),
    (zero_quantity, "Zero quantity"),
    (negative_fee, "Negative fee"),
    (extreme_notional, "Extreme notional"),
    (duplicated, "Duplicate rows")
]

for cfg in configs:
    df, title = cfg
    print(f"{title} count: {len(df)}")
    print(df[cols].head())
    print("================================\n")

# 3.F.2
df_merged["is_suspect_trade"] = np.where(
    (df_merged["price"] < 0) |
    (df_merged["quantity"] == 0) |
    (df_merged["execution_fee_usd"] < 0) |
    (df_merged.duplicated(subset=non_id_cols)),
    "Suspect Trade",
    "Normal Trade"
)

print(df_merged.groupby("is_suspect_trade")[["trade_id"]].count())


## Suggested steps
1. Profile data quality
2. Standardize categories
3. Merge dimension tables
4. Derive new features
5. Analyze liquidity, participation, market structure and P&L
6. Write an executive summary
